In [22]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [23]:
load_dotenv()

True

In [24]:
model = ChatOpenAI()

In [25]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    score: int

In [26]:
def create_outline(state: BlogState) -> BlogState:
    
    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Geneate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state

In [27]:
def create_blog(state: BlogState) -> BlogState:
    
    title = state['title']
    outline = state['outline']

    prompt = f'create a blog for the title - {title} and with outline {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state


In [32]:
def evaluate_blog(state: BlogState) -> BlogState:
    
    outline = state['outline']
    content = state['content']

    prompt = f'Based on this outline {outline} evaluate the blog content {content} and give an interger score of how good the blog is based on outline from 1-10. just give a number output'

    score = model.invoke(prompt).content

    state['score'] = score

    return state
    

In [36]:
# create graph
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [37]:
initial_state = {'title' : 'Rise of ai in india'}

final_state = workflow.invoke(initial_state)

In [38]:
print(final_state['score'])

I would rate the blog a 9 out of 10. It covers a comprehensive range of topics related to the rise of AI in India and provides a structured outline that promises an informative and engaging read for the audience.
